In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
input = 28
hidden = 128
num_layer = 2
num_classes = 10
epochs = 10
class lstm(nn.module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input,
            hidden_size=hidden,
            num_layers=num_layer,
            batch_first=True,
            dropout= 0.3
        )
        self.fc = nn.Linear(hidden, num_classes)
    def forward(self, x):
        x = x.squeeze(1)
        h0 = torch.zeros(self.num_layer, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layer, x.size(0), self.hidden_size)
        out, _ = self.lstm(x, (h0, c0))
        out, = out[:, -1, :]
        out = self.fc(out)
        return out
    
model = lstm().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
for epoch in range(epochs):
    model.train()
    total_loss = correct = total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_loss = 0.0
        correct = 0
        total = 0
        predicted   = torch.argmax(outputs, dim=1)
        correct    += (predicted == labels).sum().item()
        total      += labels.size(0)
    avg_epoch_loss = total_loss / len(train_loader)
    accuracy = correct / total * 100
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_epoch_loss:.4f}, Accuracy: {accuracy:.2f}%")
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
test_acc = correct / total * 100
print(f"Test Accuracy: {test_acc:.2f}%")
torch.save(model.state_dict(), 'lstm_mnist.pth')
print("Model saved as lstm_mnist.pth")
